# Lidar frame testbench

Interactive PCAP reader to verify lidar loading **frame by frame**.

1. Edit **paths** below (or set `DATASET` to load from `datasets.json`).
2. Run **Open PCAP** once (indexing may take several minutes on USB).
3. Scrub with the slider, `FRAME_IDX`, or `±STEP` buttons.

| Panel | Source |
|-------|--------|
| Range | Native H×W reflectivity from PCAP |
| Point cloud | Top-down XY density from XYZ |
| BEV | Histogram (sync video view) |

BEV/pointcloud panels exclude zero-range returns, use staggered reflectivity aligned with XYZ, and clip display at the 99th percentile so one bright bin does not wash out the scene. Re-run **Open PCAP** + slider cells after pulling updates.

**Synced annotations** (optional): set `ANNOTATIONS_PATH` to a JSON file keyed by **sync pair** index. Boxes use lateral Y / forward X in meters and appear on pointcloud + BEV. Use the **Annotate** cell to click two corners and save.

In [ ]:
from pathlib import Path

# --- Edit these, or set DATASET to auto-load from datasets.json ---
DATASET = "2026.05.10/18-05-08"  # e.g. "2026.05.10/18-05-08", or None

LIDAR_PCAP = "/Volumes/Seagate Portable Drive/data_collect_mobile/2026.05.10/lidar/20260510_2127_59_OS-0-128_122519000479.pcap"
LIDAR_METADATA = "/Volumes/Seagate Portable Drive/data_collect_mobile/2026.05.10/lidar/20260510_2127_59_OS-0-128_122519000479.json"

# Optional: highlight a lidar_idx from sync_pairs.csv
SYNC_CSV = Path("res/2026.05.10/18-05-08/sync_pairs.csv")

# Synced object boxes (pair index -> lateral/forward meters). None to disable.
ANNOTATIONS_PATH = Path("res/2026.05.10/18-05-08/annotations.json")
SHOW_ANNOTATIONS = True

if DATASET:
    import sys

    _here = Path.cwd()
    _sync_dir = next((c for c in [_here, *_here.parents] if (c / "datasets.json").is_file()), None)
    if _sync_dir is None:
        _sync_dir = next((c / "sync" for c in [_here, *_here.parents] if (c / "sync" / "datasets.json").is_file()), _here)
    _src_dir = _sync_dir / "src"  # pipeline modules live in code/sync/src/
    if str(_src_dir) not in sys.path:
        sys.path.insert(0, str(_src_dir))
    from lib.dataset_config import load_dataset_paths

    paths = load_dataset_paths(DATASET)
    print(f"dataset: {DATASET}")
    LIDAR_PCAP = paths["lidar_pcap"]
    LIDAR_METADATA = paths["lidar_metadata"]
    SYNC_CSV = Path(paths.get("sync_csv", SYNC_CSV))

print("PCAP:", LIDAR_PCAP)
print("Meta:", LIDAR_METADATA)
print("Sync CSV:", SYNC_CSV)

In [ ]:
import sys
import time
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np

# Run notebook from code/ directory
CODE_ROOT = Path.cwd()
if not (CODE_ROOT / "sync").is_dir() and (CODE_ROOT.parent / "sync").is_dir():
    CODE_ROOT = CODE_ROOT.parent
if str(CODE_ROOT / "sync") not in sys.path:
    sys.path.insert(0, str(CODE_ROOT / "sync"))

from lib.lidar_io import (
    close_source,
    get_ouster_api,
    get_scan_at_index,
    open_pcap_scan_source,
    scan_fingerprint,
    scan_source_length,
    sensor_info_from_source,
)
from lib.sync_annotations import (
    boxes_for_pair,
    build_sync_pair_maps,
    draw_topdown_boxes,
    load_annotations,
    pair_idx_from_lidar,
    pick_box_two_clicks,
    save_annotations,
    upsert_box,
)
from lib.bev_render import (
    lidar_panel_axis,
    lidar_panel_color_limits,
    lidar_panel_for_imshow,
    scan_to_bev,
    scan_to_pointcloud_panel,
    scan_to_range_panel,
)

print("Ouster API:", get_ouster_api())
print("Working dir:", CODE_ROOT)

In [ ]:
# --- Open PCAP (run once; slow on external USB) ---
_t0 = time.time()
_source = open_pcap_scan_source(LIDAR_PCAP, LIDAR_METADATA, index=True)
_metadata = sensor_info_from_source(_source)
N_SCANS = scan_source_length(_source)
print(f"Indexed {N_SCANS} scans in {time.time() - _t0:.1f}s")

# Load sync CSV lidar indices (optional reference)
SYNC_LIDAR_IDX = []
if Path(SYNC_CSV).is_file():
    import csv
    with open(SYNC_CSV) as f:
        SYNC_LIDAR_IDX = [int(r["lidar_idx"]) for r in csv.DictReader(f)]
    print(f"sync_pairs lidar_idx: {min(SYNC_LIDAR_IDX)} .. {max(SYNC_LIDAR_IDX)} ({len(SYNC_LIDAR_IDX)} pairs)")
else:
    print("No sync CSV found; scrub any index 0 .. N_SCANS-1")

_lidar_to_pair: dict = {}
_pair_to_lidar: dict = {}
_annotations = load_annotations(None)
if Path(SYNC_CSV).is_file():
    _lidar_to_pair, _radar_to_pair, _pair_to_lidar = build_sync_pair_maps(SYNC_CSV)
    if SHOW_ANNOTATIONS:
        _annotations = load_annotations(ANNOTATIONS_PATH)
        n = sum(len(v) for v in _annotations.get("objects", {}).values())
        print(f"annotations: {ANNOTATIONS_PATH} ({n} boxes)")


def get_scan(frame_idx: int):
    """Random access by sync-order scan index (sequential nth, not __getitem__)."""
    frame_idx = int(frame_idx)
    if frame_idx < 0 or (N_SCANS is not None and frame_idx >= N_SCANS):
        raise IndexError(f"frame_idx {frame_idx} out of range [0, {N_SCANS})")
    t0 = time.time()
    scan = get_scan_at_index(_source, frame_idx)
    print(f"  loaded scan {frame_idx} in {time.time() - t0:.2f}s")
    return scan


def render_panels(frame_idx: int):
    scan = get_scan(frame_idx)
    fp = scan_fingerprint(scan)
    range_img = scan_to_range_panel(_metadata, scan)
    pc_img = scan_to_pointcloud_panel(_metadata, scan)
    bev_img = scan_to_bev(_metadata, scan)
    return {
        "range": range_img,
        "pointcloud": pc_img,
        "bev": bev_img,
        "fp": fp,
    }

In [ ]:
def show_frame(frame_idx: int, *, compare_prev: bool = True):
    """Plot range / pointcloud / BEV for one scan. Set compare_prev=False for faster loads."""
    global _last_frame, _last_panels
    if "N_SCANS" not in globals() or N_SCANS is None:
        raise RuntimeError("Run the 'Open PCAP' cell first.")

    panels = render_panels(frame_idx)
    fp = panels["fp"]
    ts = fp.get("timestamp_s")
    ts_str = datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC") if ts else "n/a"

    pair_idx = pair_idx_from_lidar(_lidar_to_pair, frame_idx) if SHOW_ANNOTATIONS else None
    ann_boxes = boxes_for_pair(_annotations, pair_idx) if pair_idx is not None else []

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    views = [
        ("range", "Range reflectivity (H×W)"),
        ("pointcloud", "Point cloud XY density"),
        ("bev", "BEV reflectivity"),
    ]
    for ax, (key, title) in zip(axes, views):
        img = lidar_panel_for_imshow(panels[key], key)
        vmin, vmax = lidar_panel_color_limits(panels[key], key)
        axis = lidar_panel_axis(key)
        im = ax.imshow(
            img,
            origin=axis["origin"],
            extent=axis["extent"],
            aspect=axis["aspect"],
            cmap="viridis",
            vmin=vmin,
            vmax=vmax,
        )
        if axis["extent"] is not None:
            ax.set_xlim(axis["extent"][0], axis["extent"][1])
            ax.set_ylim(axis["extent"][2], axis["extent"][3])
        if key in ("pointcloud", "bev") and ann_boxes:
            draw_topdown_boxes(ax, ann_boxes)
        ax.set_title(title)
        ax.set_xlabel(axis["xlabel"])
        ax.set_ylabel(axis["ylabel"])
        plt.colorbar(im, ax=ax, fraction=0.046)

    diff_line = ""
    if compare_prev and "_last_panels" in globals() and _last_frame is not None:
        for key in ("range", "pointcloud", "bev"):
            d = float(np.mean(np.abs(panels[key] - _last_panels[key])))
            diff_line += f"  mean|Δ{key}|={d:.3f}"
    pair_line = f"  pair={pair_idx}" if pair_idx is not None else ""
    fig.suptitle(
        f"lidar_idx={frame_idx}{pair_line}  cksum={fp['range_checksum']}  "
        f"nonzero_range={fp['range_nonzero']}  t={ts_str}{diff_line}",
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()

    _last_frame = int(frame_idx)
    _last_panels = panels
    return panels


# --- Manual frame index: edit and re-run this cell ---
FRAME_IDX = 10
show_frame(FRAME_IDX)

In [ ]:
# --- Annotate: click two corners on pointcloud, save to ANNOTATIONS_PATH ---
# Tip: use %matplotlib widget if clicks do not register in Jupyter.


def annotate_lidar_pair(
    pair_idx: int,
    *,
    label: str = "object",
    box_id: str = "",
    color: str = "yellow",
):
    """Add one synced box for *pair_idx* (uses that pair's lidar_idx)."""
    global _annotations
    if not _pair_to_lidar:
        raise RuntimeError("Run Open PCAP cell first (need sync pair maps).")
    lidar_idx = _pair_to_lidar[int(pair_idx)]
    panels = render_panels(lidar_idx)
    img = lidar_panel_for_imshow(panels["pointcloud"], "pointcloud")
    axis = lidar_panel_axis("pointcloud")
    vmin, vmax = lidar_panel_color_limits(panels["pointcloud"], "pointcloud")

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(
        img,
        origin=axis["origin"],
        extent=axis["extent"],
        aspect=axis["aspect"],
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
    )
    if axis["extent"] is not None:
        ax.set_xlim(axis["extent"][0], axis["extent"][1])
        ax.set_ylim(axis["extent"][2], axis["extent"][3])
    draw_topdown_boxes(ax, boxes_for_pair(_annotations, pair_idx))
    ax.set_title(f"Annotate pair={pair_idx}  lidar_idx={lidar_idx}")
    ax.set_xlabel(axis["xlabel"])
    ax.set_ylabel(axis["ylabel"])
    plt.tight_layout()
    plt.show()

    box = pick_box_two_clicks(ax, label=label, box_id=box_id or f"pair{pair_idx}_{label}", color=color)
    if box is None:
        return None
    _annotations = upsert_box(_annotations, pair_idx, box)
    save_annotations(_annotations, ANNOTATIONS_PATH)
    print(f"Saved to {ANNOTATIONS_PATH}: pair {pair_idx} -> {box}")
    return box


# Example: annotate_lidar_pair(189, label="vehicle")

In [ ]:
# --- Interactive slider (requires ipywidgets) ---
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    print("Install ipywidgets for slider: pip install ipywidgets")
else:
    max_idx = int(N_SCANS) - 1 if N_SCANS else 0
    start_idx = SYNC_LIDAR_IDX[0] if SYNC_LIDAR_IDX else 0

    slider = widgets.IntSlider(
        value=min(start_idx, max_idx),
        min=0,
        max=max_idx,
        step=1,
        description="lidar_idx",
        continuous_update=False,
        layout=widgets.Layout(width="80%"),
    )
    out = widgets.interactive_output(show_frame, {"frame_idx": slider})
    display(widgets.VBox([slider, out]))

In [ ]:
# --- Quick probe: compare first / mid / last sync lidar_idx ---
_last = int(N_SCANS) - 1
probe = sorted({0, _last // 2, _last})
if SYNC_LIDAR_IDX:
    probe = sorted({SYNC_LIDAR_IDX[0], SYNC_LIDAR_IDX[len(SYNC_LIDAR_IDX) // 2], SYNC_LIDAR_IDX[-1]})

rows = []
for idx in probe:
    scan = get_scan(idx)
    fp = scan_fingerprint(scan)
    rows.append((idx, fp["range_checksum"], fp["range_nonzero"], fp.get("timestamp_s")))
print("idx | range_checksum | nonzero | timestamp_s")
for r in rows:
    print(f"{r[0]:5d} | {r[1]:15d} | {r[2]:7d} | {r[3]}")
if len({r[1] for r in rows}) < 2:
    print("WARN: identical checksums — PCAP indexing may be stuck")
else:
    print("OK: checksums vary across frames")

In [ ]:
# Cleanup when done (optional)
if "_source" in globals() and _source is not None:
    close_source(_source)
    _source = None
    print("PCAP closed")